# SIM V3 · Indoor · Phase B — FDTD dataset

Generate the full-wave (FDTD) training boxes for the U-Net surrogate.
Ground truth = complex field `U(x)` from `FullWaveScene`, phase-reduced to
`Ũ = U·e^{+jkd}`. Indoor: the whole 7th-floor plane per Tx (cheap fields).

Output: `fw_data_indoor/shard_*.npz` (`x[9,H,W]`, `y[2,H,W]`).

In [ ]:
# --- locate SIM V3 (works locally and on Colab) ---
# Colab: clone the repo, then set REPO_ROOT to it, e.g. '/content/Indoor_Walk_Test_7-7'.
REPO_ROOT = ''
import os, sys
if REPO_ROOT:
    SIMV3 = os.path.join(REPO_ROOT, 'Physics Engine', '2D', 'SIM V3')
else:
    SIMV3, d = os.path.abspath('..'), os.getcwd()
    for _ in range(6):
        if os.path.exists(os.path.join(d, '_bootstrap.py')): SIMV3 = d; break
        c = os.path.join(d, 'Physics Engine', '2D', 'SIM V3')
        if os.path.exists(os.path.join(c, '_bootstrap.py')): SIMV3 = c; break
        d = os.path.dirname(d)
assert os.path.exists(os.path.join(SIMV3, '_bootstrap.py')), f'set REPO_ROOT; not found: {SIMV3}'
sys.path.insert(0, SIMV3); os.chdir(SIMV3)
print('SIM V3 =', SIMV3)

In [ ]:
# Colab only: install deps (skip locally). torch usually preinstalled on Colab GPU.
# !pip -q install numpy scipy matplotlib tqdm onnxruntime
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

### Parameters — scale `n_tx` / `boxes_per_field` up for a real model.

In [ ]:
BANDS = ['LTE_B71_617']
SCENE = 'indoor'
N_TX = 6
BOXES_PER_FIELD = 80
BOX = 128
N_PER_WAVELENGTH = 8
REGION_M = 40   # outdoor only
import os
_PERSIST = '/content/drive/MyDrive' if os.path.isdir('/content/drive/MyDrive') else '.'
OUT = os.path.join(_PERSIST, 'fw_data_indoor')   # Drive on Colab (survives disconnects), local otherwise

### ⚡ GPU acceleration (CuPy) — optional, big speed-up
Moves **only the FDTD time loop** to the GPU; all physics setup (speed field, dt, sponge, damping, source) still comes from the CPU `FullWaveScene`, and it stays **float64**, so the field is numerically identical (a parity check vs the CPU solver runs below). Monkeypatches `fw_dataset._run_field`, so the **Generate** cell below runs on GPU. Falls back to CPU automatically if no GPU / CuPy. Select a **GPU runtime** (an A100 now actually gets used).

In [ ]:
# --- GPU FDTD (CuPy): reuse FullWaveScene setup, run only the time loop on GPU ---
import numpy as np, math
import fw_dataset
from fullwave2d import FullWaveScene

C0 = 299_792_458.0
_cpu_run_field = fw_dataset._run_field          # keep original (fallback + parity check)

USE_GPU = False
try:
    import cupy as cp
    USE_GPU = cp.cuda.runtime.getDeviceCount() > 0
except Exception:
    try:  # Colab GPU runtime usually ships CuPy; install the CUDA-12 wheel if not
        import subprocess, sys
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'cupy-cuda12x'], check=True)
        import cupy as cp
        USE_GPU = cp.cuda.runtime.getDeviceCount() > 0
    except Exception as e:
        print('CuPy unavailable -> staying on CPU:', e)

def _laplacian_gpu(u, inv_h2):                  # matches Spatial_Physics.laplacian (np.roll)
    lap = cp.zeros_like(u)
    for ax in range(u.ndim):
        lap += cp.roll(u, 1, axis=ax) + cp.roll(u, -1, axis=ax)
    lap -= 2.0 * u.ndim * u
    return lap * inv_h2

def _run_field_gpu(classes, h, tx_ij, f_mhz, crossings):
    sim = FullWaveScene(classes, h, f_mhz, tx_ij, source='cw')     # identical CPU setup
    steps = int(round(crossings * max(classes.shape) * h / C0 / sim.dt))
    dt, f0 = sim.dt, sim.f0
    omega = 2.0 * np.pi * f0
    inv_h2 = float(sim.inv_h2)
    u      = cp.asarray(sim.u)                  # move only the fields the loop touches
    u_prev = cp.asarray(sim.u_prev)
    cdt2   = cp.asarray(sim.cdt2)
    inv1pa = cp.asarray(sim._inv1pa)
    _1ma   = cp.asarray(sim._1ma)
    damp   = cp.asarray(sim.damp)
    rigid  = cp.asarray(sim.rigid)
    src    = sim.src_idx
    warmup = int(0.6 * steps)                   # same as _run_field's simulate() call
    period_steps = max(1, int(round((1.0 / f0) / dt)))
    win_start = max(warmup, steps - 2 * period_steps)             # phasor_periods = 2
    acc = cp.zeros(u.shape, cp.complex128); n_win = 0
    for k in range(steps):                      # mirrors FullWaveScene.step() exactly
        lap = _laplacian_gpu(u, inv_h2)
        u_next = (2.0 * u - _1ma * u_prev + cdt2 * lap) * inv1pa
        u_next[src] += math.sin(omega * (k * dt))                # cw() soft source, t=k*dt
        u_next[rigid] = 0.0                     # perfect reflectors
        u_next *= damp                          # absorbing sponge
        u_prev = u * damp
        u = u_next
        if k >= win_start:                      # on-the-fly single-freq DFT (u at (k+1)dt)
            acc += u * complex(np.exp(-1j * omega * (k + 1) * dt))
            n_win += 1
    if not bool(cp.isfinite(u).all()):
        raise FloatingPointError('field blew up (GPU)')
    return cp.asnumpy((2.0 / max(n_win, 1)) * acc)               # complex phasor U, on CPU

if USE_GPU:
    fw_dataset._run_field = _run_field_gpu       # generate() -> _indoor_field -> this
    name = cp.cuda.runtime.getDeviceProperties(0)['name'].decode()
    # parity vs CPU on a tiny field so you can trust the port
    rng = np.random.default_rng(0)
    test = ((rng.random((96, 96)) < 0.12).astype(np.int8) * 2)   # sparse concrete
    Ucpu = _cpu_run_field(test, 0.06, (48, 48), 617.0, 1.2)
    Ugpu = _run_field_gpu(test, 0.06, (48, 48), 617.0, 1.2)
    rel = float(np.abs(Ugpu - Ucpu).max() / (np.abs(Ucpu).max() + 1e-30))
    print(f'GPU FDTD ON -> {name} | parity max|dU|/|U| = {rel:.2e} (want < 1e-6)')
else:
    print('GPU FDTD OFF -> CPU _run_field (pick a GPU runtime for the speed-up).')


### Generate (this runs FDTD — the expensive, one-time step)

In [ ]:
import fw_dataset
fw_dataset.generate(BANDS, scene=SCENE, n_tx=N_TX,
                    boxes_per_field=BOXES_PER_FIELD, box=BOX,
                    n_per_wavelength=N_PER_WAVELENGTH, region_m=REGION_M,
                    out_dir=OUT, seed=1)

### Inspect one training box (materials · |Ũ| target · log-distance)

In [ ]:
import glob, numpy as np, matplotlib.pyplot as plt
d = np.load(sorted(glob.glob(OUT + '/shard_*.npz'))[0]); X, Y = d['x'], d['y']
i = 0; fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(np.argmax(X[i, :6], 0).T, origin='lower'); ax[0].set_title('materials')
ax[1].imshow(np.hypot(Y[i, 0], Y[i, 1]).T, origin='lower'); ax[1].set_title('|U~| (target)')
ax[2].imshow(X[i, 8].T, origin='lower'); ax[2].set_title('log-distance ch'); plt.show()
print('tensors:', X.shape, Y.shape)